# Phase 13 — LangGraph checkpointing and deterministic conversation memory

This notebook demonstrates a small, deterministic two-turn conversation using LangGraph checkpointing. It deliberately keeps **conversation history** separate from **retrieved document context**. The example has no retrieval, no RAG, no LLM call, no agent, no tool calling, and no routing loop.

> **Goal:** prove that the second question can use a persisted answer from the first question while the document-context field remains empty and untouched.


## Concept map

| Concept | Demonstration in this notebook |
| --- | --- |
| State | A typed **MemoryState** carries the current question, last answer, append-only conversation history, and a separate retrieved-document-context field. |
| Checkpointing | **MemorySaver** persists the state by one stable thread ID. |
| Conversation memory | The second invocation sees the first turn through the checkpointed **conversation_history** field. |
| Document context separation | **retrieved_document_context** remains an independent empty list. The response node never reads it. |
| Deterministic behavior | A fixed Python node maps the two demonstration questions to answers; no model or dynamic decision is used. |

The fixed topology is **START → respond → END**. The same compiled graph is invoked twice with the same **thread_id** so that the second turn receives the saved state.


In [1]:
from __future__ import annotations

import json
import operator
from pathlib import Path
from typing import Annotated, Literal, TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("/home/ubuntu/business-knowledge-ai")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROCESSED_DIR / "introduction_to_business_memory_results.json"
STATUS_PATH = PROCESSED_DIR / "introduction_to_business_memory_status.json"

print("Project root:", PROJECT_ROOT)
print("LangGraph MemorySaver imported successfully.")


Project root: /home/ubuntu/business-knowledge-ai
LangGraph MemorySaver imported successfully.


## 1. Define a state with strict boundaries

The fields below have distinct responsibilities. **conversation_history** is the only memory field and has an append reducer, so every invocation retains the earlier turns. **retrieved_document_context** is a different field with no reducer and no connection to the response node. It is intentionally initialized as an empty list because this learning exercise must not invent or use source documents.


In [2]:
class ConversationTurn(TypedDict):
    turn: int
    question: str
    answer: str
    answer_basis: Literal["current_turn_instruction", "checkpointed_conversation_history"]


class RetrievedDocument(TypedDict):
    chunk_id: str
    text: str
    source: str


class MemoryState(TypedDict, total=False):
    current_question: str
    answer: str
    answer_basis: str
    turn_number: int
    conversation_history: Annotated[list[ConversationTurn], operator.add]
    retrieved_document_context: list[RetrievedDocument]
    document_context_status: str


STATE_FIELD_BOUNDARIES = {
    "conversation_history": "Checkpointed append-only dialogue turns; used by the second question.",
    "retrieved_document_context": "Separate retrieval payload; empty and never read in this memory-only demonstration.",
    "current_question": "The question supplied for the current invocation.",
    "answer": "The deterministic answer for the current invocation.",
}

print(json.dumps(STATE_FIELD_BOUNDARIES, indent=2))


{
  "conversation_history": "Checkpointed append-only dialogue turns; used by the second question.",
  "retrieved_document_context": "Separate retrieval payload; empty and never read in this memory-only demonstration.",
  "current_question": "The question supplied for the current invocation.",
  "answer": "The deterministic answer for the current invocation."
}


## 2. Build a deterministic response node

The first question establishes a phrase to remember. The second question explicitly asks for the answer to the first question. The node reads only **conversation_history** on the follow-up turn. It does not inspect **retrieved_document_context**, call an LLM, retrieve documents, select a tool, or decide whether to loop.


In [3]:
QUESTION_1 = "For this checkpointing demonstration, what phrase should we remember as the business focus?"
QUESTION_2 = "Based on your previous answer, what was the business focus?"
EXPECTED_FIRST_ANSWER = "For this demonstration, the business focus is customer value."
EXPECTED_FOLLOW_UP_ANSWER = "Based on the previous answer, the business focus was customer value."


def respond(state: MemoryState) -> dict:
    history = state.get("conversation_history", [])
    question = state["current_question"].strip()
    turn_number = len(history) + 1

    # This node intentionally never reads state["retrieved_document_context"].
    if question == QUESTION_1:
        answer = EXPECTED_FIRST_ANSWER
        answer_basis = "current_turn_instruction"
    elif question == QUESTION_2:
        previous_answer = history[-1]["answer"] if history else ""
        if previous_answer == EXPECTED_FIRST_ANSWER:
            answer = EXPECTED_FOLLOW_UP_ANSWER
        else:
            answer = "I do not have the first-turn answer in this checkpoint."
        answer_basis = "checkpointed_conversation_history"
    else:
        answer = "This deterministic learning graph only supports the two documented questions."
        answer_basis = "current_turn_instruction"

    turn: ConversationTurn = {
        "turn": turn_number,
        "question": question,
        "answer": answer,
        "answer_basis": answer_basis,
    }
    return {
        "conversation_history": [turn],
        "answer": answer,
        "answer_basis": answer_basis,
        "turn_number": turn_number,
        "document_context_status": "separate_empty_context_not_read_by_memory_node",
    }


builder = StateGraph(MemoryState)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)

checkpointer = MemorySaver()
memory_graph = builder.compile(checkpointer=checkpointer)

print("Compiled deterministic graph: START -> respond -> END")


Compiled deterministic graph: START -> respond -> END


## 3. Run Question 1 and inspect the persisted state

The initial payload makes the context boundary visible: the graph receives an empty **retrieved_document_context**, while its dialogue history starts empty. The checkpoint configuration gives the conversation a stable thread ID.


In [4]:
thread_id = "phase13-memory-demo"
config = {"configurable": {"thread_id": thread_id}}

turn_1_input: MemoryState = {
    "current_question": QUESTION_1,
    "conversation_history": [],
    "retrieved_document_context": [],
    "document_context_status": "empty_no_retrieval_in_memory_demo",
}
turn_1_state = memory_graph.invoke(turn_1_input, config=config)

print("Question 1:", QUESTION_1)
print("Answer 1:", turn_1_state["answer"])
print("History after Question 1:", json.dumps(turn_1_state["conversation_history"], indent=2))
print("Retrieved-document context after Question 1:", turn_1_state["retrieved_document_context"])

assert turn_1_state["answer"] == EXPECTED_FIRST_ANSWER
assert len(turn_1_state["conversation_history"]) == 1
assert turn_1_state["retrieved_document_context"] == []


Question 1: For this checkpointing demonstration, what phrase should we remember as the business focus?
Answer 1: For this demonstration, the business focus is customer value.
History after Question 1: [
  {
    "turn": 1,
    "question": "For this checkpointing demonstration, what phrase should we remember as the business focus?",
    "answer": "For this demonstration, the business focus is customer value.",
    "answer_basis": "current_turn_instruction"
  }
]
Retrieved-document context after Question 1: []


## 4. Run Question 2 against the same thread

Only the follow-up question is supplied in the second input. The checkpoint restores the first-turn conversation history. The node answers from that history and the document-context field remains distinct and empty.


In [5]:
turn_2_state = memory_graph.invoke({"current_question": QUESTION_2}, config=config)

print("Question 2:", QUESTION_2)
print("Answer 2:", turn_2_state["answer"])
print("Answer basis:", turn_2_state["answer_basis"])
print("Conversation history after Question 2:", json.dumps(turn_2_state["conversation_history"], indent=2))
print("Retrieved-document context after Question 2:", turn_2_state["retrieved_document_context"])

assert turn_2_state["answer"] == EXPECTED_FOLLOW_UP_ANSWER
assert turn_2_state["answer_basis"] == "checkpointed_conversation_history"
assert len(turn_2_state["conversation_history"]) == 2
assert turn_2_state["retrieved_document_context"] == []


Question 2: Based on your previous answer, what was the business focus?
Answer 2: Based on the previous answer, the business focus was customer value.
Answer basis: checkpointed_conversation_history
Conversation history after Question 2: [
  {
    "turn": 1,
    "question": "For this checkpointing demonstration, what phrase should we remember as the business focus?",
    "answer": "For this demonstration, the business focus is customer value.",
    "answer_basis": "current_turn_instruction"
  },
  {
    "turn": 2,
    "question": "Based on your previous answer, what was the business focus?",
    "answer": "Based on the previous answer, the business focus was customer value.",
    "answer_basis": "checkpointed_conversation_history"
  }
]
Retrieved-document context after Question 2: []


## 5. Inspect checkpoint snapshots

**MemorySaver** stores snapshots under the thread ID. State history shows the graph's checkpoints, while the final saved values prove that dialogue memory and retrieved-document context are separate state fields.


In [6]:
checkpoint_snapshots = list(memory_graph.get_state_history(config))
persisted_state = memory_graph.get_state(config).values

print("Checkpoint snapshot count:", len(checkpoint_snapshots))
print("Persisted conversation turns:", len(persisted_state["conversation_history"]))
print("Persisted retrieved-document context:", persisted_state["retrieved_document_context"])
print("Latest persisted answer:", persisted_state["answer"])

assert len(checkpoint_snapshots) >= 4
assert persisted_state["conversation_history"] == turn_2_state["conversation_history"]
assert persisted_state["retrieved_document_context"] == []


Checkpoint snapshot count: 6
Persisted conversation turns: 2
Persisted retrieved-document context: []
Latest persisted answer: Based on the previous answer, the business focus was customer value.


## 6. Save auditable learning artifacts

The results record captures only actual graph outputs and state-boundary checks. It contains no fabricated document context or model output. The status record explicitly records the non-agentic scope.


In [7]:
results = {
    "phase": "13_memory",
    "notebook": "notebooks/12_memory.ipynb",
    "graph_topology": ["START", "respond", "END"],
    "thread_id": thread_id,
    "state_field_boundaries": STATE_FIELD_BOUNDARIES,
    "question_1": {
        "question": QUESTION_1,
        "answer": turn_1_state["answer"],
        "answer_basis": turn_1_state["answer_basis"],
        "conversation_history_count": len(turn_1_state["conversation_history"]),
        "retrieved_document_context_count": len(turn_1_state["retrieved_document_context"]),
    },
    "question_2": {
        "question": QUESTION_2,
        "answer": turn_2_state["answer"],
        "answer_basis": turn_2_state["answer_basis"],
        "conversation_history_count": len(turn_2_state["conversation_history"]),
        "retrieved_document_context_count": len(turn_2_state["retrieved_document_context"]),
        "contextual_follow_up_correct": turn_2_state["answer"] == EXPECTED_FOLLOW_UP_ANSWER,
    },
    "checkpointing": {
        "checkpointer": "MemorySaver",
        "snapshot_count": len(checkpoint_snapshots),
        "persisted_history_count": len(persisted_state["conversation_history"]),
        "persisted_document_context_count": len(persisted_state["retrieved_document_context"]),
    },
    "retrieved_document_context": {
        "provided": False,
        "count": len(persisted_state["retrieved_document_context"]),
        "read_by_memory_node": False,
        "note": "No documents were retrieved or fabricated for this memory-only demonstration.",
    },
}

status = {
    "phase": "13_memory",
    "status": "completed",
    "notebook_executed": True,
    "langgraph_checkpointing_executed": True,
    "checkpointer": "MemorySaver",
    "same_thread_two_turns_executed": True,
    "contextual_follow_up_correct": results["question_2"]["contextual_follow_up_correct"],
    "conversation_history_separate_from_retrieved_document_context": True,
    "retrieved_document_context_count": results["retrieved_document_context"]["count"],
    "retrieval_implemented": False,
    "rag_implemented": False,
    "llm_invocation_implemented": False,
    "tool_calling_implemented": False,
    "autonomous_agents_implemented": False,
    "agentic_control_flow_implemented": False,
    "retrieval_loops_implemented": False,
    "self_correction_loops_implemented": False,
}

RESULTS_PATH.write_text(json.dumps(results, indent=2) + "\n", encoding="utf-8")
STATUS_PATH.write_text(json.dumps(status, indent=2) + "\n", encoding="utf-8")

print("Saved:", RESULTS_PATH)
print("Saved:", STATUS_PATH)
print(json.dumps(status, indent=2))


Saved: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_memory_results.json
Saved: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_memory_status.json
{
  "phase": "13_memory",
  "status": "completed",
  "notebook_executed": true,
  "langgraph_checkpointing_executed": true,
  "checkpointer": "MemorySaver",
  "same_thread_two_turns_executed": true,
  "contextual_follow_up_correct": true,
  "conversation_history_separate_from_retrieved_document_context": true,
  "retrieved_document_context_count": 0,
  "retrieval_implemented": false,
  "rag_implemented": false,
  "llm_invocation_implemented": false,
  "tool_calling_implemented": false,
  "autonomous_agents_implemented": false,
  "agentic_control_flow_implemented": false,
  "retrieval_loops_implemented": false,
  "self_correction_loops_implemented": false
}


## What this establishes

The second response is contextual because **MemorySaver** restored the first turn for the same thread ID. The execution asserts that there are exactly two persisted conversation turns and zero persisted retrieved-document records. This is memory mechanics only, not a RAG implementation and not an agent.
